# Qdrant Vector Database Exercises for AI Engineering

## Learning Goal

By the end of this notebook, you will understand how to use **Qdrant** as a vector database for real AI engineering workflows.

You will learn:

- What a vector database is
- What Qdrant stores internally
- How collections, points, vectors, payloads, and filters work
- How to create a small vector search system
- How this maps to RAG, ingestion, APIs, evaluation, and production ML pipelines

### Background & Links

Useful official resources:

- Qdrant documentation: https://qdrant.tech/documentation/
- Qdrant Python client: https://python-client.qdrant.tech/
- Qdrant GitHub: https://github.com/qdrant/qdrant
- Qdrant Python client GitHub: https://github.com/qdrant/qdrant-client

### Why Qdrant matters in real AI engineering

In real AI systems, especially **RAG systems**, you usually do not send the whole document collection to an LLM.

Instead, you:

1. Split documents into chunks
2. Convert chunks into embeddings
3. Store embeddings in a vector database
4. Search for the most relevant chunks for a user question
5. Send only the retrieved chunks to the LLM

Qdrant is one of the tools that can store and search those embeddings efficiently.

## Mental Model

Think of Qdrant as a **semantic search database**.

A normal database answers:

> "Find rows where `book = 'Arabic RAG Guide'`."

A vector database answers:

> "Find text chunks that are semantically similar to this question."

### Big-picture mental model

```text
Document chunk
      ↓
Embedding model
      ↓
Vector: [0.12, -0.44, 0.88, ...]
      ↓
Qdrant collection
      ↓
Similarity search
      ↓
Top-k relevant chunks
      ↓
LLM answer with citations
```

### How this fits production systems

In production, Qdrant usually sits between:

- **Ingestion pipeline**: reads, cleans, chunks, embeds, and stores data
- **Retriever API**: receives a query, embeds it, searches Qdrant
- **Generator / LLM layer**: answers using retrieved chunks
- **Evaluation pipeline**: checks if retrieval returns the correct sources

For a RAG system, Qdrant is not the LLM.  
It is the **retrieval memory layer**.

## Background Explanation

### Core concepts

#### 1. Vector

A vector is a list of numbers that represents meaning.

Example:

```python
[0.12, 0.44, -0.31, 0.90]
```

In real systems, vectors often have hundreds or thousands of dimensions.

Examples:

- BGE-M3 embeddings: often 1024 dimensions
- OpenAI text embeddings: often 1536 or 3072 dimensions depending on model
- MiniLM embeddings: often 384 dimensions

#### 2. Collection

A collection is like a table in Qdrant.

It has:

- A name
- A vector size
- A distance metric
- Many stored points

Example:

```text
collection: arabic_books_chunks
vector_size: 1024
distance: cosine
```

#### 3. Point

A point is one stored item.

A point contains:

- `id`: unique identifier
- `vector`: embedding
- `payload`: metadata

Example:

```python
{
    "id": 1,
    "vector": [0.1, 0.2, 0.3],
    "payload": {
        "text": "This is a document chunk.",
        "book": "My Book",
        "page": 12
    }
}
```

#### 4. Payload

Payload is metadata attached to a vector.

In RAG, payload often stores:

- chunk text
- source filename
- page number
- book title
- author
- section title
- language
- document type

#### 5. Distance metric

Qdrant needs a way to compare vectors.

Common metrics:

- `COSINE`: common for text embeddings
- `DOT`: useful for some embedding models
- `EUCLID`: geometric distance

For most text-embedding RAG systems, start with **cosine similarity** unless the embedding model recommends something else.

## Setup

This notebook uses **Qdrant in-memory mode**.

That means:

- No Docker needed
- No Qdrant server needed
- Good for exercises and learning
- Not persistent after notebook restart

For production, you would usually run Qdrant using:

- Docker
- Qdrant Cloud
- Kubernetes
- A managed deployment

In [2]:
# Install Qdrant client if needed.
# Run this cell once in a fresh environment.

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("qdrant_client") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "qdrant-client"])
else:
    print("qdrant-client is already installed.")

qdrant-client is already installed.


In [3]:
from qdrant_client import QdrantClient
from qdrant_client import models

print("Qdrant client imported successfully.")

c:\AI\environments\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qdrant client imported successfully.


## Minimal Working Example

In this example, we will:

1. Create an in-memory Qdrant client
2. Create a collection
3. Insert simple vectors
4. Search for similar vectors

To keep the example simple, we will use small manual vectors.

In real AI engineering, these vectors would come from an embedding model.

In [3]:
# 1. Create an in-memory Qdrant client
client = QdrantClient(":memory:")

COLLECTION_NAME = "mini_ai_docs"

# 2. Create a collection
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE,
    ),
)

print(f"Collection created: {COLLECTION_NAME}")

Collection created: mini_ai_docs


C:\Users\abdul\AppData\Local\Temp\ipykernel_15680\188029241.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [4]:
# 3. Insert points into Qdrant

points = [
    models.PointStruct(
        id=1,
        vector=[0.90, 0.10, 0.10, 0.00],
        payload={
            "text": "RAG retrieves relevant chunks before calling the LLM.",
            "topic": "rag",
            "source": "rag_notes.md",
        },
    ),
    models.PointStruct(
        id=2,
        vector=[0.85, 0.15, 0.05, 0.00],
        payload={
            "text": "Vector databases store embeddings and support similarity search.",
            "topic": "vector_db",
            "source": "qdrant_notes.md",
        },
    ),
    models.PointStruct(
        id=3,
        vector=[0.05, 0.10, 0.90, 0.10],
        payload={
            "text": "FastAPI exposes backend endpoints for applications.",
            "topic": "api",
            "source": "api_notes.md",
        },
    ),
]

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

print("Inserted points:", len(points))

Inserted points: 3


In [5]:
# 4. Search for similar vectors

query_vector = [0.88, 0.12, 0.08, 0.00]

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=2,
)

for point in results.points:
    print("ID:", point.id)
    print("Score:", round(point.score, 4))
    print("Text:", point.payload["text"])
    print("Topic:", point.payload["topic"])
    print("-" * 60)

ID: 1
Score: 0.9995
Text: RAG retrieves relevant chunks before calling the LLM.
Topic: rag
------------------------------------------------------------
ID: 2
Score: 0.9987
Text: Vector databases store embeddings and support similarity search.
Topic: vector_db
------------------------------------------------------------


### Short explanation

The query vector is close to points about **RAG** and **vector databases**, so Qdrant returns those first.

In a real RAG pipeline:

- The user question becomes a query embedding
- Stored chunks already have embeddings
- Qdrant compares query embedding against stored chunk embeddings
- The best chunks are returned

## Exercise 1

### Goal

Create your own Qdrant collection for a tiny AI knowledge base.

You will:

1. Create a collection named `ai_concepts`
2. Use vector size `3`
3. Insert at least 4 points
4. Search using a query vector
5. Print the retrieved texts

### TODO section

Complete the code below.

### Questions to think about

- Why must the collection vector size match the vector size of each point?
- What happens if you insert a vector with the wrong length?
- Why is payload useful in RAG systems?

In [15]:
# Exercise 1 TODO

exercise_client = QdrantClient(":memory:")
EXERCISE_COLLECTION = "ai_concepts"

# TODO 1:
# Create a collection with:
# - collection_name = EXERCISE_COLLECTION
# - vector size = 3
# - distance = COSINE

exercise_client.recreate_collection(
    collection_name=EXERCISE_COLLECTION,
    vectors_config=models.VectorParams(
        size=3,
        distance=models.Distance.COSINE),
    )
                                    


# TODO 2:
# Insert at least 4 points.
# Each point should have:
# - id
# - vector of length 3
# - payload with "text" and "category"

exercise_points = [
    models.PointStruct(id=1,
                       vector=[0.1,0.2,0.3],
                       payload={
                           "text": "This is going to be the testing text 1",
                           "category": "Fiction"}),
    models.PointStruct(id=2,
                       vector=[0.5,0.6,0.7],
                       payload={
                           "text": "This is going to be the testing text 2",
                           "category": "Romance"}),
    models.PointStruct(id=3,
                       vector=[0.7,0.8,0.9],
                       payload={
                           "text": "This is going to be the testing text 3",
                           "category": "Daram"}),
                           
]

exercise_client.upsert(collection_name=EXERCISE_COLLECTION,
                       points=exercise_points)


# TODO 3:
# Search with a query vector of length 3 and limit=2

results = exercise_client.query_points(collection_name=EXERCISE_COLLECTION,
                                       query = [0.1,0.2,0.4],
                                       limit=2)

# TODO 4:
# Print the retrieved payload text and score
for point in results.points:
    print(point.id)
    print(point.payload)

1
{'text': 'This is going to be the testing text 1', 'category': 'Fiction'}
2
{'text': 'This is going to be the testing text 2', 'category': 'Romance'}


C:\Users\abdul\AppData\Local\Temp\ipykernel_15680\439358292.py:12: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  exercise_client.recreate_collection(


## Solution 1

In [ ]:
# Solution 1

exercise_client = QdrantClient(":memory:")
EXERCISE_COLLECTION = "ai_concepts"

exercise_client.recreate_collection(
    collection_name=EXERCISE_COLLECTION,
    vectors_config=models.VectorParams(
        size=3,
        distance=models.Distance.COSINE,
    ),
)

exercise_points = [
    models.PointStruct(
        id=1,
        vector=[0.95, 0.05, 0.05],
        payload={
            "text": "Embeddings convert text into numeric vectors.",
            "category": "embeddings",
        },
    ),
    models.PointStruct(
        id=2,
        vector=[0.90, 0.10, 0.05],
        payload={
            "text": "Vector search retrieves semantically similar content.",
            "category": "retrieval",
        },
    ),
    models.PointStruct(
        id=3,
        vector=[0.05, 0.95, 0.10],
        payload={
            "text": "APIs expose model functionality to applications.",
            "category": "api",
        },
    ),
    models.PointStruct(
        id=4,
        vector=[0.10, 0.05, 0.95],
        payload={
            "text": "Evaluation checks whether the system returns correct answers.",
            "category": "evaluation",
        },
    ),
]

exercise_client.upsert(
    collection_name=EXERCISE_COLLECTION,
    points=exercise_points,
)

query_vector = [0.92, 0.08, 0.05]

results = exercise_client.query_points(
    collection_name=EXERCISE_COLLECTION,
    query=query_vector,
    limit=2,
)

for point in results.points:
    print("Score:", round(point.score, 4))
    print("Text:", point.payload["text"])
    print("Category:", point.payload["category"])
    print("-" * 60)

## Exercise 2

### Slightly harder

Now add **payload filtering**.

In real AI systems, we often do not want to search all documents.

Examples:

- Search only Arabic documents
- Search only one book
- Search only chunks from a specific stage
- Search only public documents
- Search only documents belonging to one user or tenant

### Goal

Create a collection with mixed documents and retrieve only documents from a selected category.

You will:

1. Create a collection named `filtered_ai_docs`
2. Insert points with payload field `category`
3. Search for similar vectors
4. Filter results where `category = "rag"`

### Questions to think about

- Why is filtering important in production RAG?
- What can go wrong if you forget tenant/user filters?
- Why should source metadata be stored in payload?

In [16]:
# Exercise 2 TODO

filter_client = QdrantClient(":memory:")
FILTER_COLLECTION = "filtered_ai_docs"

# TODO 1:
# Create a collection with vector size 4 and COSINE distance.
filter_client.recreate_collection(
    collection_name=FILTER_COLLECTION,
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE,
    ),
)

# TODO 2:
# Insert points with payload fields:
# - text
# - category
# - source

points = [
    models.PointStruct(
        id=0,
        vector=[1,2,3,4],
        payload={
            "text":"this is 1",
            "category": "RAG"
        }
    ),
    models.PointStruct(
        id=1,
        vector=[2,3,4,5],
        payload={
            "text":"this is 2",
            "category": "RAG"
        }
    ),
    models.PointStruct(
        id=2,
        vector=[3,4,5,6],
        payload={
            "text":"this is 3",
            "category": "Finetuning"
        }
    ),
    models.PointStruct(
        id=3,
        vector=[4,5,6,7],
        payload={
            "text":"this is 4",
            "category": "RAG"
        }
    )
]

filter_client.upsert(FILTER_COLLECTION, points=points)


# TODO 3:
# Create a Qdrant filter that only allows category == "rag"
results = filter_client.query_points(
    collection_name=FILTER_COLLECTION,
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="category",
                match=models.MatchValue(value="RAG")
            )
        ]
    ),
    limit=1
)

# TODO 4:
# Search using query_points with query_filter=your_filter

# TODO 5:
# Print results
for result in results.points:
    print(result.payload)

{'text': 'this is 1', 'category': 'RAG'}


C:\Users\abdul\AppData\Local\Temp\ipykernel_42292\3725916909.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  filter_client.recreate_collection(


## Solution 2

In [ ]:
# Solution 2

filter_client = QdrantClient(":memory:")
FILTER_COLLECTION = "filtered_ai_docs"

filter_client.recreate_collection(
    collection_name=FILTER_COLLECTION,
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE,
    ),
)

filter_points = [
    models.PointStruct(
        id=1,
        vector=[0.90, 0.10, 0.05, 0.00],
        payload={
            "text": "RAG uses retrieval to ground LLM answers in source documents.",
            "category": "rag",
            "source": "rag_intro.md",
        },
    ),
    models.PointStruct(
        id=2,
        vector=[0.88, 0.12, 0.08, 0.00],
        payload={
            "text": "A retriever returns top-k chunks before generation.",
            "category": "rag",
            "source": "retrieval_notes.md",
        },
    ),
    models.PointStruct(
        id=3,
        vector=[0.10, 0.85, 0.10, 0.05],
        payload={
            "text": "FastAPI can expose a search endpoint to users.",
            "category": "api",
            "source": "api_notes.md",
        },
    ),
    models.PointStruct(
        id=4,
        vector=[0.05, 0.10, 0.90, 0.05],
        payload={
            "text": "Evaluation datasets contain questions, expected answers, and sources.",
            "category": "evaluation",
            "source": "eval_notes.md",
        },
    ),
]

filter_client.upsert(
    collection_name=FILTER_COLLECTION,
    points=filter_points,
)

rag_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="category",
            match=models.MatchValue(value="rag"),
        )
    ]
)

query_vector = [0.89, 0.11, 0.06, 0.00]

filtered_results = filter_client.query_points(
    collection_name=FILTER_COLLECTION,
    query=query_vector,
    query_filter=rag_filter,
    limit=5,
)

for point in filtered_results.points:
    print("Score:", round(point.score, 4))
    print("Category:", point.payload["category"])
    print("Text:", point.payload["text"])
    print("Source:", point.payload["source"])
    print("-" * 60)

## Common Mistakes

### Mistake 1: Vector dimension mismatch

If the collection expects vectors of size `4`, every inserted vector must have length `4`.

Wrong:

```python
vector=[0.1, 0.2, 0.3]
```

Correct:

```python
vector=[0.1, 0.2, 0.3, 0.4]
```

### Mistake 2: Forgetting payload

If you store only vectors, retrieval becomes hard to use.

Bad point:

```python
{"id": 1, "vector": [...]}
```

Better point:

```python
{
    "id": 1,
    "vector": [...],
    "payload": {
        "text": "...",
        "source": "book.pdf",
        "page": 12
    }
}
```

### Mistake 3: Using random vectors in production

Manual vectors are fine for learning.

In production, use an embedding model such as:

- BGE-M3
- multilingual-e5
- sentence-transformers
- OpenAI embeddings
- Cohere embeddings
- Jina embeddings

### Mistake 4: Using different embedding models for indexing and querying

This is a serious retrieval bug.

Bad:

```text
Store chunks using model A
Search questions using model B
```

Good:

```text
Store chunks using BGE-M3
Search questions using BGE-M3
```

### Mistake 5: Forgetting filters in multi-user systems

If your system has multiple users, tenants, or private documents, always filter by ownership.

Example payload:

```python
{
    "tenant_id": "company_123",
    "user_id": "user_456"
}
```

Then always filter search by `tenant_id` or `user_id`.

In [ ]:
# Debugging example: vector dimension mismatch

debug_client = QdrantClient(":memory:")
DEBUG_COLLECTION = "debug_dimension_mismatch"

debug_client.recreate_collection(
    collection_name=DEBUG_COLLECTION,
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE,
    ),
)

try:
    debug_client.upsert(
        collection_name=DEBUG_COLLECTION,
        points=[
            models.PointStruct(
                id=1,
                vector=[0.1, 0.2, 0.3],  # Wrong length: expected 4
                payload={"text": "This vector has the wrong size."},
            )
        ],
    )
except Exception as e:
    print("Expected error:")
    print(type(e).__name__)
    print(e)

## Real AI Engineering Usage

### 1. RAG

Qdrant stores embedded chunks.

Typical payload:

```python
{
    "text": "chunk text",
    "book": "Arabic RAG Guide",
    "page_start": 15,
    "page_end": 16,
    "section": "Chunking Strategy"
}
```

The retriever searches Qdrant and returns the best chunks for the LLM.

### 2. APIs

A FastAPI backend might expose:

```text
POST /search
POST /ask
POST /documents/ingest
```

Internally, those endpoints call Qdrant.

Example flow:

```text
User question
   ↓
API receives request
   ↓
Embedding model creates query vector
   ↓
Qdrant returns top-k chunks
   ↓
LLM generates grounded answer
```

### 3. ML pipelines

Qdrant can support:

- semantic deduplication
- dataset search
- nearest-neighbor inspection
- clustering workflows
- similarity-based labeling

### 4. Ingestion

During ingestion, you usually:

1. Load files
2. Clean text
3. Split into chunks
4. Embed each chunk
5. Upsert points into Qdrant

### 5. Evaluation

In RAG evaluation, you can test:

- Did Qdrant retrieve the expected source?
- Is the expected page in top-k?
- Did retrieval return irrelevant chunks?
- Are filters working correctly?

### 6. Vector DBs

Qdrant is one vector database option.

Other tools include:

- Chroma
- Weaviate
- Milvus
- Pinecone
- Elasticsearch vector search
- PostgreSQL with pgvector

Qdrant is often a strong choice when you want:

- open-source deployment
- strong metadata filtering
- good Python support
- production-oriented vector search

## Final Mini Exercise

### Small production-style task

Build a tiny retrieval layer for an AI project.

You will create a function:

```python
search_knowledge_base(query_vector, category=None, limit=3)
```

The function should:

1. Search a Qdrant collection
2. Optionally filter by category
3. Return clean Python dictionaries with:
   - score
   - text
   - category
   - source

This is similar to what you would build inside a real RAG retriever module.

In [22]:
# Final Mini Exercise TODO

prod_client = QdrantClient(":memory:")
PROD_COLLECTION = "production_style_kb"

prod_client.recreate_collection(
    collection_name=PROD_COLLECTION,
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE,
    ),
)

prod_points = [
    models.PointStruct(
        id=1,
        vector=[0.90, 0.10, 0.05, 0.00],
        payload={
            "text": "RAG answers should be grounded in retrieved source chunks.",
            "category": "rag",
            "source": "rag_quality.md",
        },
    ),
    models.PointStruct(
        id=2,
        vector=[0.86, 0.14, 0.07, 0.01],
        payload={
            "text": "Citations help users verify generated answers.",
            "category": "rag",
            "source": "citations.md",
        },
    ),
    models.PointStruct(
        id=3,
        vector=[0.10, 0.85, 0.08, 0.02],
        payload={
            "text": "FastAPI is commonly used to serve AI backend endpoints.",
            "category": "api",
            "source": "fastapi.md",
        },
    ),
    models.PointStruct(
        id=4,
        vector=[0.05, 0.10, 0.90, 0.05],
        payload={
            "text": "Evaluation checks retrieval quality and answer faithfulness.",
            "category": "evaluation",
            "source": "evaluation.md",
        },
    ),
]

prod_client.upsert(
    collection_name=PROD_COLLECTION,
    points=prod_points,
)

# TODO:
# Implement this function.
def search_knowledge_base(query_vector, category=None, limit=3):
    filter = models.Filter(
        must=[
            models.FieldCondition(
                key="category",
                match=models.MatchValue(value=category)
            )
             ]
         )
    
    results = prod_client.query_points(
        collection_name=PROD_COLLECTION,
        query_filter=filter,
        query=query_vector,
        limit=limit
        )

    for point in results.points:
        print(point.payload)
    

# Test after implementing:
search_knowledge_base([0.88, 0.12, 0.06, 0.00], category="rag", limit=2)

{'text': 'RAG answers should be grounded in retrieved source chunks.', 'category': 'rag', 'source': 'rag_quality.md'}
{'text': 'Citations help users verify generated answers.', 'category': 'rag', 'source': 'citations.md'}


C:\Users\abdul\AppData\Local\Temp\ipykernel_42292\3699322434.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  prod_client.recreate_collection(


### Final Mini Exercise Solution

In [ ]:
# Final Mini Exercise Solution

def search_knowledge_base(query_vector, category=None, limit=3):
    query_filter = None

    if category is not None:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="category",
                    match=models.MatchValue(value=category),
                )
            ]
        )

    results = prod_client.query_points(
        collection_name=PROD_COLLECTION,
        query=query_vector,
        query_filter=query_filter,
        limit=limit,
    )

    clean_results = []

    for point in results.points:
        clean_results.append(
            {
                "score": point.score,
                "text": point.payload.get("text"),
                "category": point.payload.get("category"),
                "source": point.payload.get("source"),
            }
        )

    return clean_results


search_knowledge_base(
    query_vector=[0.88, 0.12, 0.06, 0.00],
    category="rag",
    limit=2,
)

## Key Takeaways

- Qdrant is a vector database used for similarity search.
- A collection is like a table for vectors.
- A point contains an ID, vector, and payload.
- Payload stores useful metadata such as text, source, page, author, and category.
- The vector size in the collection must match the vector size of inserted points.
- For text RAG, cosine similarity is a common starting point.
- In production, vectors should come from a real embedding model, not manual lists.
- Use the same embedding model for indexing and querying.
- Filters are critical for source selection, user permissions, tenant isolation, and better retrieval.
- In RAG, Qdrant is the retrieval memory layer, not the generator.